# Chapter 8 — Human in the Loop

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

<!-- provider-note:start -->
## Setting the backbone LLM of your agent

These notebooks run on **Ollama by default**, the setup the book teaches.
If you do nothing, nothing changes: `make_llm()` starts a local Ollama
service when one isn't already running.

To use OpenAI or Anthropic instead:

1. Install the extra: `uv sync --extra openai` or `uv sync --extra anthropic`.
2. Export `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` before launching Jupyter.
3. Set `LLM_PROVIDER=openai` (or `anthropic`), or pass `provider="openai"`
   to `make_llm()`. A key on its own never switches providers, so one
   exported for unrelated work cannot reroute you off the Ollama path.

The switch applies wherever a notebook builds its LLM with `make_llm()`.
A notebook that constructs `OllamaLLM` directly stays on Ollama regardless
of these settings.

If you opted in but forgot to export the key, you will be prompted for it.
That is the safe path on hosted kernels, and it keeps the key out of the
saved notebook. Setting `OLLAMA_API_KEY` alone routes Ollama to Ollama
Cloud.

**Caveat:** the examples are tuned for qwen3. Output on gpt-5 or Claude
will differ from what is printed in the book, and prompt-sensitive
examples (the ch09 evaluator pattern, ch08 supervised trajectories) may
behave noticeably differently.
<!-- provider-note:end -->

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [2]:
from llm_agents_from_scratch.notebook_utils import make_llm

/home/nerdai/Projects/llm-agents-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Examples

### Example 1: Agent-Initiated Pause with HumanInputTool

In [3]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.tools import SimpleFunctionTool
from llm_agents_from_scratch.tools.default import HumanInputTool

enable_console_logging(logging.INFO)


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)
human_input_tool = HumanInputTool()
llm = make_llm()
agent = LLMAgent(llm=llm, tools=[next_number_tool, human_input_tool])

✓ Using Ollama Cloud (deepseek-v4.1-flash)


In [4]:
task = Task(
    instruction=(
        "Compute the full Hailstone sequence step by step using next_number. "
        "The starting number must come from the human operator — ask them "
        "before beginning."
    )
)
result = await agent.run(task)
print(result.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the full Hailstone sequence step by step using next_number. The starting number must come from the human operator — ask them ...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the full Hailstone sequence step by step using next_number. The starting number must come from the human operator — ask th...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__human_input


╭────────────────────────────────────────────────── Human Input ──────────────────────────────────────────────────╮
│ What starting number should I use to compute the Hailstone sequence?                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

>:

[scripted human] > 6
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 6
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "d9e1130e-d152-441d-83d9-95c6d6ce8c59",
    "tool_name": "from_scratch__use_skill...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Execute the tool call `from_scratch__use_skill` with arguments `{"name": "stop-at-one"}` to load the stop-at-one skill, then proceed to co...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Execute the tool call `from_scratch__use_skill` with arguments `{"name": "stop-at-one"}` to load the stop-at-one skill, then proce...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_skill
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: <skill_content name="stop-at-one">
# Stop At One

Compute a full sequence from a starting number down to 1
using the `next_num...[TRUNCATED]
INFO 

### Example 2: End-of-Loop Approval Gate

In [5]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.tools import SimpleFunctionTool

enable_console_logging(logging.INFO)


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)
llm = make_llm()
agent = LLMAgent(llm=llm, tools=[next_number_tool])

✓ Using Ollama Cloud (deepseek-v4.1-flash)


In [6]:
task = Task(
    instruction=(
        "Compute the full Hailstone sequence starting from 5, "
        "step by step using next_number, until you reach 1."
    )
)
result = await agent.run(task, with_approval=True)
print(result.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the full Hailstone sequence starting from 5, step by step using next_number, until you reach 1.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the full Hailstone sequence starting from 5, step by step using next_number, until you reach 1.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_skill
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: <skill_content name="stop-at-one">
# Stop At One

Compute a full sequence from a starting number down to 1
using the `next_num...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "4a47b8b5-f9f2-4f16-afb7-ba9d3766266a",
    "tool_name": "next_number",
    "argu...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Call the next_number tool with x=5 and wait for the result before continuing.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Process

╭───────────────────────────────────────────── Proposed Task Result ──────────────────────────────────────────────╮
│ The sequence has reached 1, so it's complete. Let me report the result.                                         │
│                                                                                                                 │
│ The complete Hailstone sequence starting from 5 is:                                                             │
│                                                                                                                 │
│ **5 → 16 → 8 → 4 → 2 → 1**                                                                                      │
│                                                                                                                 │
│ - **Starting number:** 5                                                                                        │
│ - **Total steps taken:** 5 tool calls                                                                           │
│ - **Maximum value reached:** 16                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Approve this result? [y/n]:

[scripted human] > y
INFO (llm_agents_fs.LLMAgent) :      🏁 Task completed: The sequence has reached 1, so it's complete. Let me report the result.

The complete Hailstone sequence starting from 5 is:

**5 → ...[TRUNCATED]
The sequence has reached 1, so it's complete. Let me report the result.

The complete Hailstone sequence starting from 5 is:

**5 → 16 → 8 → 4 → 2 → 1**

- **Starting number:** 5
- **Total steps taken:** 5 tool calls
- **Maximum value reached:** 16


### Example 3: Caller-Driven Stepwise Execution with run_supervised()

In [7]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.tools import SimpleFunctionTool

enable_console_logging(logging.INFO)


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)
llm = make_llm()
agent = LLMAgent(llm=llm, tools=[next_number_tool])

✓ Using Ollama Cloud (deepseek-v4.1-flash)


In [8]:
task = Task(
    instruction=(
        "Compute the full Hailstone sequence starting from 4, "
        "step by step using next_number, until you reach 1."
    )
)
task_handler = await agent.run_supervised(task, explicit_only_skills=["stop-at-one"])

In [9]:
step = await task_handler.get_next_step(None)
step

TaskStep(id_='0fb7474c-3c21-47a9-833b-38bef4047200', task_id='370e9631-9440-45d4-8ed9-80b4b88a27f6', instruction='Compute the full Hailstone sequence starting from 4, step by step using next_number, until you reach 1.')

In [10]:
step_result = await task_handler.run_step(step)
step_result

INFO (llm_agents_fs.SupervisedTaskHandler) :      ⚙️ Processing Step: Compute the full Hailstone sequence starting from 4, step by step using next_number, until you reach 1.
INFO (llm_agents_fs.SupervisedTaskHandler) :      🛠️ Executing Tool Call: next_number
INFO (llm_agents_fs.SupervisedTaskHandler) :      ✅ Successful Tool Call: 2
INFO (llm_agents_fs.SupervisedTaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "9b59844e-1d30-4caa-817c-75a37326ec9e",
    "tool_name": "next_number",
    "argu...[TRUNCATED]


TaskStepResult(task_step_id='0fb7474c-3c21-47a9-833b-38bef4047200', content='I need to make the following tool-calls:\n{\n    "id_": "9b59844e-1d30-4caa-817c-75a37326ec9e",\n    "tool_name": "next_number",\n    "arguments": {\n        "x": 2\n    }\n}')

In [11]:
step = await task_handler.get_next_step(step_result)
step

INFO (llm_agents_fs.SupervisedTaskHandler) :      🧠 New Step: Call next_number with x=2 to continue the Hailstone sequence from 4: 4 → 2 → ?


TaskStep(id_='3e9d24a4-cd8d-4d04-8c04-7289b158dcb2', task_id='370e9631-9440-45d4-8ed9-80b4b88a27f6', instruction='Call next_number with x=2 to continue the Hailstone sequence from 4: 4 → 2 → ?')

In [12]:
step_result = await task_handler.run_step(step)
step_result

INFO (llm_agents_fs.SupervisedTaskHandler) :      ⚙️ Processing Step: Call next_number with x=2 to continue the Hailstone sequence from 4: 4 → 2 → ?
INFO (llm_agents_fs.SupervisedTaskHandler) :      🛠️ Executing Tool Call: next_number
INFO (llm_agents_fs.SupervisedTaskHandler) :      ✅ Successful Tool Call: 1
INFO (llm_agents_fs.SupervisedTaskHandler) :      ✅ Step Result: Continuing the sequence: 4 → 2 → 1.

Since we've reached 1, the full Hailstone sequence starting from 4 is complete:

**4 → 2 → 1**

Th...[TRUNCATED]


TaskStepResult(task_step_id='3e9d24a4-cd8d-4d04-8c04-7289b158dcb2', content="Continuing the sequence: 4 → 2 → 1.\n\nSince we've reached 1, the full Hailstone sequence starting from 4 is complete:\n\n**4 → 2 → 1**\n\nThat's the entire sequence — it terminates after 3 terms, which makes sense since 4 is a power of 2 (4 = 2²), so it divides down to 1 in exactly 2 steps.")

In [13]:
step = await task_handler.get_next_step(step_result)
step

INFO (llm_agents_fs.SupervisedTaskHandler) :      No new step required.


TaskResult(task_id='370e9631-9440-45d4-8ed9-80b4b88a27f6', content="Continuing the sequence: 4 → 2 → 1.\n\nSince we've reached 1, the full Hailstone sequence starting from 4 is complete:\n\n**4 → 2 → 1**\n\nThat's the entire sequence — it terminates after 3 terms, which makes sense since 4 is a power of 2 (4 = 2²), so it divides down to 1 in exactly 2 steps.")

In [14]:
# since result is a TaskResult, we can call complete()
await task_handler.complete(step)
print(result.content)

The sequence has reached 1, so it's complete. Let me report the result.

The complete Hailstone sequence starting from 5 is:

**5 → 16 → 8 → 4 → 2 → 1**

- **Starting number:** 5
- **Total steps taken:** 5 tool calls
- **Maximum value reached:** 16


---

> **Note — Agent Inspector**
>
> The agent above can also be driven step-by-step through a browser UI using
> the **Agent Inspector**
> ([`llm-agents-from-scratch-inspector`](https://github.com/nerdai/llm-agents-from-scratch-inspector)).
> Install it with `pip install llm-agents-from-scratch-inspector`, then create
> a launch script that exposes an `agent_builder` at module scope:

```python
# main.py
from llm_agents_from_scratch import LLMAgentBuilder
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.notebook_utils import make_llm
from llm_agents_from_scratch.tools import SimpleFunctionTool


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


agent_builder = (
    LLMAgentBuilder()
    .with_llm(make_llm())
    .with_tool(SimpleFunctionTool(func=next_number))
)

default_task = Task(
    instruction=(
        "Compute the full Hailstone sequence starting from 4, "
        "step by step using next_number, until you reach 1."
    )
)
```

> Then launch:
>
> ```bash
> agent-inspector launch main.py
> ```
>
> `make_llm()` starts a local Ollama server if one isn't already running. To
> point the agent at Ollama Cloud instead, set `OLLAMA_API_KEY` in your
> environment; to use OpenAI or Anthropic, set `LLM_PROVIDER` as described in
> the provider note at the top of this notebook. The Inspector works the same
> either way.
>
> A browser tab opens where you can enter a task, create a session, and step
> through `get_next_step()` / `run_step()` interactively. See the
> [Inspector repository](https://github.com/nerdai/llm-agents-from-scratch-inspector)
> for full usage instructions.